# Boston City Data Cleaner from MassDOT only
# (Expected Runtime: 2-5 minutes)

Output in /data folder:

-boston-metro2.csv - contains MassDOT crash records for specified regions, formatted for Mass Crash Map

# Importing all libraries

In [1]:
!pip install pandas
!pip install numpy
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Collecting MassDOT crash data for BOSTON CITY and years (update in 2027)

In [2]:
all_features = []

#Inserted years individually because "2023v" naming issue in MassDOT crash server
#https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT

years = ['2021', '2022','2023v','2024','2025','2026']

for year in years:
   
    base_url = f"https://gis.crashdata.dot.mass.gov/arcgis/rest/services/MassDOT/MASSDOT_ODP_OPEN_{year}/FeatureServer/0/query"
    
    params = {
        "where": " CITY_TOWN_NAME = 'BOSTON' ",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "returnGeometry": "true",
        "resultOffset": 0,
        "resultRecordCount": 2000
    }

    while True:
        
        response = requests.get(base_url, params=params)
        data = response.json()
        features = data.get("features", [])
        
        if not features:
            break
        
        all_features.extend(features)
        params["resultOffset"] += params["resultRecordCount"]

records = [f["attributes"] for f in all_features]

df = pd.DataFrame(records)
print("Shape:", df.shape)
df.head()

Shape: (27503, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATE_TEXT,CRASH_TIME_2,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,4915103,BOSTON,01 02 2021,7:38 PM,1.609634e+12,07:00PM to 07:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),1,...,NaN,Interstate,4616461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4915101,BOSTON,01 02 2021,9:13 AM,1.609597e+12,09:00AM to 09:59AM,Closed,Non-fatal injury,Suspected Minor Injury (B),1,...,NaN,Interstate,4616463,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4915422,BOSTON,01 04 2021,3:05 PM,1.609791e+12,03:00PM to 03:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Interstate,4617214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4915421,BOSTON,01 03 2021,9:22 PM,1.609727e+12,09:00PM to 09:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Principal Arterial - Other,4617215,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4916193,BOSTON,01 01 2021,2:40 PM,1.609530e+12,02:00PM to 02:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,...,NaN,Local,4617630,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#CRASH_TIME_2 & CRASH_DATE_TEXT columns are of type str
#LAT & LON are of type float64
#The following code ensures time and date are of type datetime and lat and lon are numeric
df["CRASH_TIME"] = pd.to_datetime(df["CRASH_TIME_2"], format="%I:%M %p", errors="coerce").dt.time
df["CRASH_DATE"] = pd.to_datetime(df["CRASH_DATE_TEXT"], errors="coerce")
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df = df.drop(columns=["CRASH_DATE_TEXT", "CRASH_TIME_2"])
mass_crashes = df
print("Shape of MASSDOT dataset:", mass_crashes.shape)
mass_crashes.head()

Shape of MASSDOT dataset: (27503, 124)


,CRASH_NUMB,CITY_TOWN_NAME,CRASH_DATETIME,CRASH_HOUR,CRASH_STATUS,CRASH_SEVERITY_DESCR,MAX_INJR_SVRTY_CL,NUMB_VEHC,NUMB_NONFATAL_INJR,NUMB_FATAL_INJR,...,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL,CRASH_TIME,CRASH_DATE
0,4915103,BOSTON,1.609634e+12,07:00PM to 07:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),1,0,0,...,4616461,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19:38:00,2021-01-02
1,4915101,BOSTON,1.609597e+12,09:00AM to 09:59AM,Closed,Non-fatal injury,Suspected Minor Injury (B),1,1,0,...,4616463,NaN,NaN,NaN,NaN,NaN,NaN,NaN,09:13:00,2021-01-02
2,4915422,BOSTON,1.609791e+12,03:00PM to 03:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4617214,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15:05:00,2021-01-04
3,4915421,BOSTON,1.609727e+12,09:00PM to 09:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4617215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21:22:00,2021-01-03
4,4916193,BOSTON,1.609530e+12,02:00PM to 02:59PM,Closed,Property damage only (none injured),No Apparent Injury (O),2,0,0,...,4617630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14:40:00,2021-01-01


# Standardizing Different Types of Vulnerable Roadway Users

Goal - Since we aim to create a Massachusetts Crash Map that is easy to understand, we have recoded over 20 different types of vulnerable roadway users that appear in MassDOT crash data reports. While MassDOT intended vulernable user types to be standardized (such as "Pedestrians" and "Bicyclists"), in practice the data is pulled from various police reports across the states, with varying levels of quality and uniformity. Also, labels for different devices are ambiguous. For example, current MA General Law uses the term "Motorized Bicyclist" to mean a "moped" (with a gas motor), not an e-bike. 

Data Limitations - The MassDOT crash reports include two fields were police officers may select from a drop-down menu of various types of vulnerable users, or harmful events that involved vulnerable users:

* NON_MTRST_TYPE_CL (Non-Motorist Type), shown as column1 below
* MOST_HRMFL_EVT_CL (Most Harmful Event), shown as column2 below

TODO for README - recreate as table with column1 terms (Bicyclist) vs column2 terms (Collision with Cyclist), etc
  
However, the MassDOT crash records drawn from local police may not be internally consistent. For example, a crash report may list "Pedestrian" as a Non-Motorist Type in column 1, but NOT include "Collision with pedestrian" as Most Harmful Event in column 2, and vice versa. 

Method: While there is no perfect method, we decided to condense similar types of vulnerable users (as listed in MassDOT crash records) into three broad categories, based on their GENERAL SPEED (not their actual speed during specific crashes):

Pedestrians - and related users who generally travel at LOW speeds
* Pedestrian in column1, OR Collision with pedestrian in column2
* Electric Personal Assistive Mobility Device User
* Non-Motorized Wheelchair User    TODO check for other types of wheelchairs
* Emergency Responder - Outside of vehicle
* Roadway Worker - Outside of vehicle
* Utility Worker - Outside of vehicle

Cyclists - and related micromobility users who generally travel at MEDIUM speeds
* Bicyclist in column1, OR Collision with cyclist (bicycle, tricycle, unicycle, pedal car) in column2
* Hand Cyclist
* Inline Skater
* Non-Motorized Scooter Rider
* Other Micromobility Device User
* Roller Skater
* Skateboarder
* Tricyclist

Other vulnerable users who generally travel at HIGHER speeds OR do not fit either category above OR are unknown
* Other in column1, OR Collision with Other Vulnerable Users in column2
* Motorized Bicyclist in column1, OR Collision with moped in column2
* Motorized Scooter Rider
* Train/Trolley Passenger
* Farm Equipment Operator
* Unknown
  
The following code creates new columns with numerical values:
* PEDESTRIAN column = 1 if a pedestrian or similar low-speed user was involved in a crash, otherwise = 0
* CYCLIST column = 1 if a cyclist or similar micromobility user was involved in a crash, otherwise = 0
* OTHER column = 1 if higher-speed user or other type listed above or unknown was involved in a crash, otherwise = 0

We do not intend these columns to be mutually exclusive. For example, if a MassDOT crash recorded both a pedestrian and a cyclist as vulnerable users, both columns would have a value of 1. However, other crash records (such as Boston Vision Zero) are mutually exclusive and report only "ped" or "bike".


In [4]:
col1 = mass_crashes["NON_MTRST_TYPE_CL"]
col2 = mass_crashes["MOST_HRMFL_EVT_CL"]

mass_crashes["PEDESTRIAN"] = np.where(
    col1.str.contains("Pedestrian|Electric Personal Assistive Mobility Device|Wheelchair|Responder|Worker", case=False, na=False) |
    col2.str.contains("Pedestrian", case=False, na=False),
    1,
    0
)

#contains Bicyclist in column1 or Cyclist in column2 = 1, but if "Motorized" = false
mass_crashes["CYCLIST"] = np.where(
    (
        col1.str.contains("Bicyclist|Cyclist|Skater|Non-Motorized Scooter Rider|Micromobility|Skateboarder|Tricyclist", case=False, na=False) |
        col2.str.contains("Cyclist", case=False, na=False)
    )
    &
    ~(
        col1.str.contains("Motorized", case=False, na=False) |
        col2.str.contains("Motorized", case=False, na=False)
    ),
    1,
    0
)

#contains Motorized Bicyclist or Motorized Scooter or moped or Other types below
mass_crashes["OTHER"] = np.where(
    mass_crashes["NON_MTRST_TYPE_CL"].str.contains("Other|Motorized Bicyclist|Motorized Scooter Rider|Passenger|Farm|Unknown", case=False, na=False) |
    mass_crashes["MOST_HRMFL_EVT_CL"].str.contains("Other Vulnerable|moped", case=False, na=False),
    1,
    0
)

mass_crashes['SEVERITY'] = mass_crashes['CRASH_SEVERITY_DESCR'].map({
    'Fatal injury': 1,
    'Non-fatal injury': 2
}).fillna(0)


mass_crashes['INTERSTATE'] = np.where(
    mass_crashes['F_CLASS'].str.contains("Interstate", case=False, na=False),
    1,
    0
)

#trims "Local police" to "Local", "State police" to "State", etc
mass_crashes['POLICE'] = mass_crashes['POLC_AGNCY_TYPE_DESCR'].str.split().str[0]
mass_crashes['ID'] = mass_crashes['CRASH_NUMB']
mass_crashes['MUNI'] = mass_crashes['CITY_TOWN_NAME']
mass_crashes = mass_crashes.drop(columns=["CITY_TOWN_NAME", "CRASH_NUMB", "POLC_AGNCY_TYPE_DESCR"])
mass_crashes['SOURCE'] = 'MassDOT'
mass_crashes['YEAR'] = pd.to_datetime(mass_crashes["CRASH_DATE"]).dt.year


**Columns are attributes of each individual crash:**

* SOURCE: specifies where the crash data comes from(MASSDOT, Vision Zero, Somerville
PD and Cambridge PD)
* ID: unique identifier of each crash. Cambridge PD did not have an ID attribute so one
was created by adding CPD_ to the index number of each crash.
* MUNI: specifies the area each crash occurred(Boston, Cambridge, Brookline,
Somerville)
* YEAR: the year of each crash in format YYYY (for quicker data quality checks)
* DATE: the date each crash occurred in this format YEAR-MONTH-DAY
* SEVERITY: it is 1 if the crash was fatal, 2 if the crash resulted in nonfatal injuries and
blank if neither
* CRASH_TIME: time each crash occured
* POLICE: specifies if Local, State, MBTA or Campus police reported the crash
* PEDESTRIAN: 1 if pedestrian was involved in the crash, 0 if not
* CYCLIST: 1 if cyclist OR micromobility user was involved in the crash, 0 if not
* OTHER: 1 if other type of vulnerable user was involved in the crash, 0 if not
* LAT: latitude of the location where the crash occurred
* LON: longitude of the location where the crash occurred
* INTERSTATE: 1 if the crash occurred on an interstate highway, 0 if not

In [5]:
cols_to_move = [
    "SOURCE",
    
    "ID",

    "MUNI",

    "YEAR",

    "CRASH_DATE",

    "SEVERITY",

    "CRASH_TIME",

    "POLICE",

    "PEDESTRIAN",

    "CYCLIST",

    "OTHER",

    "LAT",

    "LON",

    "INTERSTATE",

    "NON_MTRST_TYPE_CL",

    "MOST_HRMFL_EVT_CL"

    
]
df = mass_crashes[cols_to_move + [c for c in mass_crashes.columns if c not in cols_to_move]]
df.head()

,SOURCE,ID,MUNI,YEAR,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,PEDESTRIAN,CYCLIST,...,T_EXC_TIME,F_F_CLASS,OBJECTID,TRAFFIC_CONTROL_TYPE_DESCR,NON_MTRST_ORIGIN_DEST_CL,NON_MTRST_CNTRB_CIRC_CL,NON_MTRST_DISTRACTED_BY_CL,NON_MTRST_ALC_SUSPD_TYPE_CL,NON_MTRST_DRUG_SUSPD_TYPE_CL,NON_MTRST_EVENT_SEQ_CL
0,MassDOT,4915103,BOSTON,2021,2021-01-02,0.0,19:38:00,State,0,0,...,NaN,Interstate,4616461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MassDOT,4915101,BOSTON,2021,2021-01-02,2.0,09:13:00,State,0,0,...,NaN,Interstate,4616463,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MassDOT,4915422,BOSTON,2021,2021-01-04,0.0,15:05:00,State,0,0,...,NaN,Interstate,4617214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MassDOT,4915421,BOSTON,2021,2021-01-03,0.0,21:22:00,State,0,0,...,NaN,Principal Arterial - Other,4617215,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MassDOT,4916193,BOSTON,2021,2021-01-01,0.0,14:40:00,State,0,0,...,NaN,Local,4617630,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# UPDATE INDEX AS NEEDED: We keep only relevant columns and drop rows that don't have a date
df_short = df.iloc[:, :16]  ## because added two string columns at end
df_short = df_short.dropna(subset=["CRASH_DATE"])

In [7]:
#TODO Fix interstate error below
# We make sure numerical data is of type int or float, and strings of type str
df_short["CRASH_DATE"] = pd.to_datetime(df_short["CRASH_DATE"]).dt.date
df_short['SEVERITY'] = pd.to_numeric(df_short['SEVERITY'], errors='coerce').astype('Int64')
df_short['INTERSTATE'] = df_short['INTERSTATE'].astype(str)
df_short['ID'] = df_short['ID'].astype(str)

In [8]:
df_short.info()

<class 'pandas.DataFrame'>
RangeIndex: 27503 entries, 0 to 27502
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   SOURCE             27503 non-null  str    
 1   ID                 27503 non-null  str    
 2   MUNI               27503 non-null  str    
 3   YEAR               27503 non-null  int32  
 4   CRASH_DATE         27503 non-null  object 
 5   SEVERITY           27503 non-null  Int64  
 6   CRASH_TIME         27500 non-null  object 
 7   POLICE             27501 non-null  object 
 8   PEDESTRIAN         27503 non-null  int64  
 9   CYCLIST            27503 non-null  int64  
 10  OTHER              27503 non-null  int64  
 11  LAT                25003 non-null  float64
 12  LON                25003 non-null  float64
 13  INTERSTATE         27503 non-null  str    
 14  NON_MTRST_TYPE_CL  1539 non-null   str    
 15  MOST_HRMFL_EVT_CL  23977 non-null  str    
dtypes: Int64(1), float64(2), int32(1)

In [9]:
df_short.head()

,SOURCE,ID,MUNI,YEAR,CRASH_DATE,SEVERITY,CRASH_TIME,POLICE,PEDESTRIAN,CYCLIST,OTHER,LAT,LON,INTERSTATE,NON_MTRST_TYPE_CL,MOST_HRMFL_EVT_CL
0,MassDOT,4915103,BOSTON,2021,2021-01-02,0,19:38:00,State,0,0,0,NaN,NaN,1,NaN,V1:(Collision with guardrail)
1,MassDOT,4915101,BOSTON,2021,2021-01-02,2,09:13:00,State,0,0,0,42.303943,-71.052401,1,NaN,V1:(Collision with guardrail)
2,MassDOT,4915422,BOSTON,2021,2021-01-04,0,15:05:00,State,0,0,0,NaN,NaN,1,NaN,V1:(Collision with median barrier) / V2:(Colli...
3,MassDOT,4915421,BOSTON,2021,2021-01-03,0,21:22:00,State,0,0,0,42.286090,-71.043322,0,NaN,V1:(Collision with motor vehicle in traffic) /...
4,MassDOT,4916193,BOSTON,2021,2021-01-01,0,14:40:00,State,0,0,0,NaN,NaN,0,NaN,V1:(Collision with motor vehicle in traffic) /...


### Renaming columns to align with Mass Crash Map format

In [10]:
renaming = {
    "LAT": "lat",
    "LON": "lng",
    "CRASH_DATE": "date",
    "CRASH_TIME": "time",
    "YEAR": "year",
    "PEDESTRIAN": "pedestrian",
    "CYCLIST": "cyclist",
    "OTHER": "other",
    "INTERSTATE": "interstate",
    "SEVERITY": "severity",
    "ID": "id",
    "MUNI": "muni",
    "SOURCE": "source",
    "POLICE": "police",
    "NON_MTRST_TYPE_CL": "NON_MTRST_TYPE_CL",  #keep as is
    "MOST_HRMFL_EVT_CL": "MOST_HRMFL_EVT_CL"
}

df = df_short.rename(columns= renaming)

In [11]:
# rewrite file name to match above
df.to_csv("boston-city-from-massdot.csv")